In [ ]:
import os
import pandas as pd                                         # for data manipulation
import torch                                                # for tensor computations
from transformers import AutoTokenizer, AutoModel, GPT2LMHeadModel, GPT2Tokenizer     # for transformers
from tqdm import tqdm                                       # for progress bar
import numpy as np                                          # for numerical operations
import matplotlib.pyplot as plt                             # for plotting
import nltk                                                 # Natural Language Toolkit
from nltk.tokenize import sent_tokenize                     # for sentence tokenization
import re                                                   # for regex operations
import spacy                                                # for NLP processing
from empath import Empath                                   # for LIWC analysis
from sentence_transformers import SentenceTransformer, util  # for semantic analysis
from scipy.stats import ttest_ind                           # for statistical tests
import collections                                          # for counting duplicates
import seaborn as sns                                       # for visualizations
import random                                               # for random seed setting
import gc                                                   # for garbage collection

# Download necessary NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


SEED = 999
random.seed(SEED)
np.random.seed(SEED)

# BERT Configuration
MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 8
MAX_LENGTH = 512
CHUNK_OVERLAP = 50

# Global model and tokenizer for BERT (loaded once)
bert_tokenizer = None
bert_model = None
bert_device = None

# Feature Engineering Pipeline with BERT

Call `process_dataset_with_bert()` with your dataset path, output folder, and list of writer columns (include the human column) to run the full feature extraction pipeline including BERT embeddings.

In [8]:
# ==================== BERT EMBEDDING HELPERS ====================

def initialize_bert_model(model_name=MODEL_NAME):
    """Initialize BERT model and tokenizer once."""
    global bert_tokenizer, bert_model, bert_device
    
    if bert_model is None:
        print(f"\nLoading BERT model: {model_name}")
        bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
        bert_model = AutoModel.from_pretrained(model_name)
        
        # Freeze model weights
        for param in bert_model.parameters():
            param.requires_grad = False
        
        bert_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        bert_model.to(bert_device)
        bert_model.eval()
        print(f"Model loaded on device: {bert_device}")


def chunk_text_by_tokens(text, max_length=MAX_LENGTH, overlap=CHUNK_OVERLAP):
    """
    Split long text into overlapping chunks based on token count.
    Ensures chunks never exceed max_length tokens.
    Returns list of token tensors.
    """
    encoded = bert_tokenizer.encode_plus(
        text,
        add_special_tokens=False,
        truncation=False,
        return_tensors=None
    )
    token_ids = encoded['input_ids']
    
    # If text fits in one chunk, return it
    if len(token_ids) <= max_length - 2:  # -2 because of special tokens
        inputs = bert_tokenizer.encode_plus(
            text,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return [inputs]
    
    # Split tokens into overlapping chunks
    chunks = []
    chunk_size = max_length - 2
    stride = chunk_size - overlap
    
    for i in range(0, len(token_ids), stride):
        chunk_ids = token_ids[i:i + chunk_size]
        
        # Convert back to text
        chunk_text = bert_tokenizer.decode(chunk_ids, skip_special_tokens=True)
        
        chunk_input = bert_tokenizer.encode_plus(
            chunk_text,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        chunks.append(chunk_input)
        
        if i + chunk_size >= len(token_ids):
            break
    
    return chunks if chunks else [bert_tokenizer.encode_plus(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )]


def get_bert_embeddings(text_list, batch_size=BATCH_SIZE):
    """
    Extract BERT embeddings for long texts using token-based chunking.
    
    Args:
        text_list: List of text strings (can be very long)
        batch_size: Number of chunks to process at once    
    Returns:
        numpy array of shape (n_texts, 768) containing embeddings
    """
    all_embeddings = []
    
    for text in tqdm(text_list, desc="Extracting BERT Embeddings"):
        try:
            chunks = chunk_text_by_tokens(str(text))
            chunk_embeddings = []
            
            for i in range(0, len(chunks), batch_size):
                batch_chunks = chunks[i:i+batch_size]
                
                # Stack inputs safely
                input_ids = torch.cat([c['input_ids'] for c in batch_chunks]).to(bert_device)
                attention_mask = torch.cat([c['attention_mask'] for c in batch_chunks]).to(bert_device)
                
                with torch.no_grad():
                    outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
                    cls_embs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                    chunk_embeddings.append(cls_embs)
            
            all_chunk_embs = np.vstack(chunk_embeddings)
            text_embedding = np.mean(all_chunk_embs, axis=0)
            all_embeddings.append(text_embedding)
            
        except Exception as e:
            print(f"Error processing text: {str(e)[:100]}") 
            all_embeddings.append(np.zeros(768))  # 768 is BERT hidden size
    
    return np.array(all_embeddings)


def add_bert_features(df, output_dir,text_column='Text'):
    """
    Add BERT embeddings to a DataFrame.
    
    Args:
        df: Input DataFrame
        text_column: Name of the column containing text (default: 'Text')
    
    Returns:
        DataFrame with BERT features added
    """
    # Initialize model on first call
    initialize_bert_model()
    
    texts = df[text_column].tolist()
    
    print(f"\n{'='*60}")
    print("Adding BERT embeddings...")
    print(f"{'='*60}")
    
    bert_embeddings = get_bert_embeddings(texts)
    
    bert_columns = [f'bert_{i}' for i in range(bert_embeddings.shape[1])]
    df_bert = pd.DataFrame(bert_embeddings, columns=bert_columns)
    df_combined = pd.concat([df.reset_index(drop=True), df_bert], axis=1)
    
    print(f"\nBERT embeddings added - shape: {df_combined.shape}")
    
    # Cleanup
    del bert_embeddings, df_bert
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("BERT features added!")
    
    output_path = os.path.join(output_dir, "DB_final_with_BERT.csv")
    df_combined.to_csv(output_path, index=False)

## Usage

In [ ]:
input_files=[
    
# "data/DB_final.csv"
 "data/misspelled/DB_final.csv",
 "data/MTG_Benchmark/unseen/DB_final.csv",
 "data/MTG_Benchmark/Cross-Domain/Essay/DB_final.csv",
 "data/MTG_Benchmark/Cross-Domain/WP/DB_final.csv",
 "data/paraphrasing/nllb/DB_final.csv",
"data/paraphrasing/gemini/DB_final.csv"
]
output_files=[
    #  "data/"
    "data/misspelled/",
    "data/MTG_Benchmark/unseen/",
    "data/MTG_Benchmark/Cross-Domain/Essay/",   # separate folder
    "data/MTG_Benchmark/Cross-Domain/WP/",       # separate folder
    "data/paraphrasing/nllb/",
    "data/paraphrasing/gemini/"
]

for input_csv, output_dir in zip(input_files, output_files):
    df = pd.read_csv(input_csv)
    df_with_bert = add_bert_features(df, output_dir, text_column='Text')


Adding BERT embeddings...


Extracting BERT Embeddings: 100%|██████████| 2567/2567 [01:40<00:00, 25.65it/s]



BERT embeddings added - shape: (2567, 788)
BERT features added!

Adding BERT embeddings...


Extracting BERT Embeddings: 100%|██████████| 400/400 [00:13<00:00, 30.29it/s]



BERT embeddings added - shape: (400, 788)
BERT features added!

Adding BERT embeddings...


...


BERT embeddings added - shape: (398, 788)
BERT features added!

Adding BERT embeddings...


Extracting BERT Embeddings: 100%|██████████| 3064/3064 [01:41<00:00, 30.26it/s]



BERT embeddings added - shape: (3064, 788)
BERT features added!

Adding BERT embeddings...


Extracting BERT Embeddings: 100%|██████████| 495/495 [00:10<00:00, 45.33it/s]



BERT embeddings added - shape: (495, 788)
BERT features added!
